<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.3}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumna:} & Leon Garcia Gael Arturo \\[6pt]
\textbf{Fecha de realización:} & 25/04/2026
\end{array}

</center>

***
**UNAM no guarda relación alguna con las marcas mencionadas como ejemplo. Las marcas son propiedad de sus titulares conforme a la legislación aplicable, se utilizan con fines académicos y didácticos, por lo que no existen fines de lucro, relación publicitaria o de patrocinio.

---

#Proyecto Final: Análisis Integral del Riesgo Crediticio y Predicción del Monto de Préstamo

## Desarrollo del proyecto: pasos detallados

### Paso 1: Configuración del entorno e importación de librerías






Primero, importaremos las librerías necesarias.



In [ ]:
# Para manipulación y análisis de datos
import pandas as pd
# Para operaciones numéricas (opcional, pero útil)
import numpy as np
# Para visualización de datos
import matplotlib.pyplot as plt
# Para visualización de datos mejorada
import seaborn as sns

# Para dividir los datos en conjuntos de entrenamiento y prueba
from sklearn.model_selection import train_test_split
# Para validación cruzada y optimización de hiperparámetros [2, 43]
from sklearn.model_selection import RepeatedKFold, RandomizedSearchCV

# Modelos de Regresión
from sklearn.linear_model import LinearRegression

# Modelos de Clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Para preprocesamiento de datos
from sklearn.preprocessing import MinMaxScaler
# Alternativa a MinMaxScaler
# from sklearn.preprocessing import StandardScaler

# Para evaluación de modelos
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Para interpretación de modelos (SHAP)
import shap
import xgboost   # Para un ejemplo de modelo no lineal en SHAP

#Para la barra de avance
from tqdm import tqdm
from joblib import parallel_backend
import joblib

print("Librerías importadas correctamente.")

## Parte A: Preparación y exploración de datos


### Paso 2: Adquisición de datos financieros (LendingClub)
Utilizaremos un conjunto de datos de préstamos de LendingClub disponibles en GitHub Gist. Este dataset es ideal para problemas de riesgo crediticio.

In [ ]:
# Cargar los conjuntos de datos desde GitHub Gist y combinarlos
# Usaremos el conjunto completo mencionado en ML_Tema06 para una muestra más grande y representativa [83, 84]
dataset_urls = [ # [83]
    "https://gist.githubusercontent.com/RHDZMOTA/71c7bfc23dbd13eb8a1dfb26f7399510/raw/c0274bcb736ba2f94c29aa3d1baf2136a75f02e5/dataset-lendingclub-custom-low-funding.csv",
    "https://gist.githubusercontent.com/RHDZMOTA/71c7bfc23dbd13eb8a1dfb26f7399510/raw/c0274bcb736ba2f94c29aa3d1baf2136a75f02e5/dataset-lendingclub-custom-medium-funding.csv",
    "https://gist.githubusercontent.com/RHDZMOTA/71c7bfc23dbd13eb8a1dfb26f7399510/raw/c0274bcb736ba2f94c29aa3d1baf2136a75f02e5/dataset-lendingclub-custom-large-funding.csv"
]
df = pd.concat([pd.read_csv(url) for url in dataset_urls])

# Muestra las primeras filas del DataFrame [85]
print("Primeras 5 filas del dataset de LendingClub:")
print(df.head())

# Muestra el tamaño del DataFrame
print(f"\nEl dataset tiene {df.shape} filas y {df.shape[1]} columnas.")

In [ ]:
df.head(5)

**Pregunta de Reflexión 1**:
¿Qué representan las columnas  y  en este conjunto de datos financieros de LendingClub? ¿Cómo se relacionan con la predicción del nivel de riesgo crediticio y qué tipo de problemas de ML (regresión o clasificación) abordaríamos con cada una?


**Posible Respuesta**:
*   loan_amnt: Representa el monto del préstamo solicitado por el prestatario [Evidencia de Aprendizaje, Objetivo de Aprendizaje]. Es una variable numérica continua, lo que la hace adecuada para problemas de regresión, donde se busca predecir un valor dentro de un rango (por ejemplo, el monto que un prestatario podría solicitar o ser aprobado).
*   loan_status: Representa el estado actual del préstamo (por ejemplo, 'Fully Paid', 'Charged Off', 'Current', etc.) [Previous Response]. Es una variable categórica nominal, que se puede transformar en una variable categórica binaria (: 1 para éxito, 0 para no éxito)   para problemas de clasificación, donde se busca predecir el resultado del préstamo.
*   Ambas columnas son fundamentales para evaluar el nivel de riesgo crediticio  .  (transformado a ) es una medida directa del resultado del riesgo, mientras que  puede ser un factor que influye en el riesgo o que se predice en función del perfil de riesgo del solicitante.

### Paso 3: Exploración inicial de datos (EDA básico)
Es fundamental entender la estructura y características de nuestros datos.

In [ ]:

# Obtener información general del DataFrame (tipos de datos, valores no nulos)
print("\nInformación general del dataset:")
df.info()

# Estadísticas descriptivas de las columnas numéricas
print("\nEstadísticas descriptivas del dataset:")
print(df.describe())

# Verificar la presencia de valores nulos
print("\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0]) # Solo muestra columnas con valores nulos

**Pregunta de Reflexión 2**:
¿Qué observaciones clave puedes hacer sobre la calidad de los datos (valores faltantes, tipos de datos, rangos) que influirán en los próximos pasos de limpieza y preparación?


**Posible Respuesta**:
*   Este conjunto de datos de LendingClub sí presenta valores nulos en varias columnas, como  (duración del empleo),  (ingreso anual),  (relación deuda-ingresos) y otras [Previous Response]. Esto significa que necesitaremos aplicar técnicas de manejo de datos faltantes en el siguiente paso de limpieza.
*   Los tipos de datos () muestran una mezcla de  (para categóricas como , , , ) y numéricos (,  para , , , , ) [Previous Response]. Algunas columnas  requerirán codificación, y otras como  y  necesitarán limpieza antes de la conversión.
*   Observando , los rangos de las variables numéricas son variados, como  (500 a 35000),  (5.32% a 28.99%), y  (0 a más de 7 millones) [Previous Response]. La alta variabilidad y la presencia de valores mínimos de 0 en algunas columnas numéricas indican la necesidad de una revisión cuidadosa de los datos y la importancia del escalado para muchos modelos.


### Paso 4: Limpieza y preparación de datos avanzada


La limpieza de datos es un paso crucial en cualquier proyecto de ML, constituyendo aproximadamente el 80% de la labor de un analista de datos. En este paso, nos enfocaremos en:
1. Crear la variable objetivo de clasificación loan_success a partir de loan_status.
2. Manejar valores nulos.
3. Limpiar y ajustar tipos de datos para columnas específicas como term y emp_length.
4. Convertir las variables categóricas a un formato numérico utilizando "one-hot encoding".
5. Normalizar o estandarizar las variables numéricas, lo cual es esencial para modelos como las Redes Neuronales.

In [ ]:
# 1. Crear la variable objetivo de clasificación 'loan_success'
# Consideramos 'Fully Paid' como éxito (1) y cualquier otro estado como no éxito (0)
# Se incluyen también "Does not meet the credit policy. Status:Fully Paid" como éxito, según ML_Tema06 .
successful_statuses = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
df['loan_success'] = df['loan_status'].isin(successful_statuses).astype(int)
# Eliminar la columna original 'loan_status' ya que hemos creado 'loan_success'
df_processed = df.drop(columns=['loan_status'])

# Eliminar columnas con muchos valores nulos o irrelevantes para el modelado directo [Previous Response]
# Las columnas 'id', 'member_id', 'emp_title', 'url', 'Unnamed: 0' suelen ser identificadores o texto libre
# con alta cardinalidad que requieren procesamiento avanzado o no son directamente predictivas para este ejercicio.
df_processed = df_processed.drop(columns=['id', 'member_id', 'emp_title', 'url', 'Unnamed: 0'], errors='ignore')

# Limpiar y ajustar tipos de datos para algunas columnas antes de manejar nulos o codificar
# 'term' necesita limpieza para ser numérica (ej. ' 36 months' -> 36)
df_processed['term'] = df_processed['term'].astype(str).str.replace(' months', '', regex=False).astype(int)
# 'emp_length' necesita limpieza (ej. '< 1 year', '10+ years') a valores numéricos
df_processed['employment_length'] = df_processed['employment_length'].astype(str).str.replace(' years?', '', regex=True)
#.str.replace('< 1', '0').str.replace('10\\+', '10').astype(float)

# Eliminar filas con valores nulos restantes después de la limpieza de texto
# Esta es una estrategia simple; en un proyecto real se considerarían imputaciones más sofisticadas.
df_processed = df_processed.dropna() #

# 2. Convertir variables categóricas a numéricas usando one-hot encoding
# Identificar columnas categóricas restantes
categorical_cols = df_processed.select_dtypes(include='object').columns.tolist() #
print(f"\nColumnas categóricas a codificar con one-hot encoding: {categorical_cols}")

df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True) # drop_first=True evita multicolinealidad

# 3. Normalizar/Estandarizar las variables numéricas (MinMaxScaler)
# Identificar columnas numéricas (excluyendo 'loan_success' y 'loan_amnt' si se escalan por separado o si es la variable objetivo)
numeric_cols_to_scale = df_processed.select_dtypes(include=np.number).columns.tolist()
if 'loan_success' in numeric_cols_to_scale: numeric_cols_to_scale.remove('loan_success')
if 'loan_amnt' in numeric_cols_to_scale: numeric_cols_to_scale.remove('loan_amnt') # La variable objetivo de regresión no se escala ahora

scaler = MinMaxScaler(feature_range=(0, 1))
df_processed[numeric_cols_to_scale] = scaler.fit_transform(df_processed[numeric_cols_to_scale])

print("\nPrimeras 5 filas del dataset después de la limpieza, codificación y escalado:")
print(df_processed.head())
print("\nInformación general del dataset después de la limpieza, codificación y escalado:")
df_processed.info()


**Pregunta de Reflexión 3**:
¿Por qué la elección de estrategias para manejar valores nulos (eliminar filas vs. imputar) y el tipo de escalado (MinMaxScaler vs. StandardScaler) son decisiones importantes en la fase de preparación de datos, y cómo pueden afectar a los diferentes modelos que se utilizarán?

**Posible Respuesta**:
*   Manejo de valores nulos: La presencia de valores faltantes ( o ) puede causar errores en la ejecución de los algoritmos o llevar a resultados sesgados si no se tratan adecuadamente. Eliminar filas con nulos es simple pero puede reducir significativamente el tamaño del dataset, perdiendo información valiosa. Imputar valores (con la media, mediana o métodos más sofisticados) mantiene el tamaño del dataset pero puede introducir sesgos si la imputación no es representativa. La elección depende de la cantidad de nulos y la naturaleza de los datos.
*   Escalado de numéricas: El escalado es crucial para muchos algoritmos. Si las variables numéricas tienen rangos muy diferentes, aquellas con valores más grandes podrían dominar el proceso de aprendizaje.
*   **MinMaxScaler** transforma los datos para que se ajusten a un rango específico (ej. [86]) [29, 80]. Es útil cuando se desea mantener la forma de la distribución original.
*   **StandardScaler** (no usado aquí, pero es una alternativa común) transforma los datos para que tengan una media de cero y una desviación estándar de uno [80]. Asume que los datos siguen una distribución normal y es robusto a valores atípicos.
*   Los modelos basados en distancias (como K-NN o SVM, aunque SVM no está en las fuentes) o en gradientes (como Redes Neuronales) son particularmente sensibles a la escala de las características [28]. Los modelos basados en árboles (Árbol de Decisión, Random Forest) son menos sensibles al escalado porque sus decisiones se basan en umbrales de características, no en distancias absolutas.



### Paso 5: Análisis exploratorio de datos (EDA Visual)



Una vez preparados, podemos visualizar nuestros datos para entender mejor sus características.

In [ ]:
df.columns

In [ ]:
# Visualizar la distribución del monto del préstamo (variable de regresión)
plt.figure(figsize=(10, 5))
sns.histplot(df_processed['funded_amount'], bins=20, kde=True)
plt.title('Distribución del Monto del Préstamo (funded_amount)')
plt.xlabel('Monto del Préstamo (USD)')
plt.ylabel('Frecuencia')
plt.show()

# Visualizar la distribución del éxito del préstamo (variable de clasificación)
plt.figure(figsize=(7, 7))
df_processed['loan_success'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, colors=['lightgreen', 'lightcoral'], labels=['Préstamo Exitoso (1)', 'Préstamo No Exitoso (0)'])
plt.title('Distribución del Éxito del Préstamo (loan_success) - Nivel de Riesgo Crediticio')
plt.ylabel('')
plt.show()

# Crear un gráfico de dispersión de ingreso anual vs. monto del préstamo por éxito del préstamo
plt.figure(figsize=(10, 6))
sns.scatterplot(x='annual_income', y='funded_amount', hue='loan_success', data=df_processed, palette='coolwarm', alpha=0.6)
plt.title('Ingreso Anual vs. Monto del Préstamo por Éxito del Préstamo')
plt.xlabel('Ingreso Anual (Escalado)')
plt.ylabel('Monto del Préstamo (USD)')
plt.grid(True)
# Limitar los ejes para mejor visualización si hay outliers significativos
plt.ylim(0, df_processed['funded_amount'].quantile(0.99))
plt.xlim(0, df_processed['annual_income'].quantile(0.99))
plt.show()

# Gráfico de barras para una variable categórica codificada (ej. term_60_months)
# Esto muestra el impacto de una categoría específica en el riesgo crediticio
plt.figure(figsize=(8, 5))
sns.barplot(x='term', y='loan_success', data=df.loc[df_processed.index]) # Usamos el df original filtrado para 'term' antes de one-hot encoding para mejor interpretabilidad
plt.title('Probabilidad de Éxito del Préstamo por Plazo')
plt.xlabel('Plazo del Préstamo (meses)')
plt.ylabel('Probabilidad Media de Éxito (loan_success)')
plt.show()

# Pregunta de Reflexión 4:
Analiza los gráficos generados. ¿Qué patrones o tendencias relevantes puedes identificar en la distribución de  ingreso anual y monto de préstamo? ¿Hay alguna relación visual entre ingreso anual y monto de préstamo que sugiera factores de riesgo o éxito? ¿Qué te revela el gráfico de  sobre la probabilidad de éxito del préstamo?


**Posible Respuesta**:
*   El histograma del  muestra que la mayoría de los préstamos solicitados se concentran en ciertos montos. Es probable que haya picos en montos redondos como 5,000, 10,000, 15,000, etc., que son comunes en las solicitudes de préstamo.
*   El gráfico de pastel de  nos da una idea de la proporción de préstamos exitosos (1) frente a los no exitosos (0). Es común en datasets financieros que la mayoría de los préstamos sean pagados (clase mayoritaria), lo cual es un indicador importante del nivel de riesgo crediticio general.
*   El gráfico de dispersión de  ingreso anual y monto de préstamo, sugiere una posible relación positiva: generalmente, prestatarios con mayores ingresos anuales tienden a solicitar o a obtener préstamos de montos más altos. La diferenciación por  podría mostrar si esta relación es diferente para los préstamos exitosos y los que no lo fueron, ayudando a identificar patrones de riesgo. Por ejemplo, si se observa que los préstamos "No Exitosos" se agrupan en rangos de  más bajos y/o  más altos, esto sería un indicador de alto riesgo.
*   El gráfico de barras para  (plazo del préstamo) revelará si hay una diferencia en la probabilidad de éxito del préstamo entre plazos más cortos (ej. 36 meses) y plazos más largos (ej. 60 meses). Es común que los préstamos a plazos más largos tengan una menor probabilidad de éxito (mayor riesgo) debido a la mayor duración de la exposición al riesgo.


## Parte B: Modelado y evaluación de Regresión (predicción del monto del préstamo)


Para nuestras tareas de regresión, elegiremos la variable  como la variable dependiente (y) que queremos predecir.


### Paso 6: Preparación para los modelos de regresión


Definiremos las variables independientes (X) y la variable dependiente (y) y dividiremos el dataset en conjuntos de entrenamiento y prueba.

In [ ]:
# Definir las variables independientes (X_reg) y la variable dependiente (y_reg)
# Para la regresión, excluiremos 'loan_amnt' (nuestra variable objetivo),
# y 'loan_success' (nuestra variable objetivo para clasificación) de las características para X.
X_reg = df_processed.drop(columns=['funded_amount', 'loan_success']) # Variables de entrada
y_reg = df_processed['funded_amount'] # Variable objetivo a predecir

# Dividir los datos en conjuntos de entrenamiento y prueba para regresión
# Usaremos el 80% para entrenamiento y el 20% para prueba
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print(f"\nTamaño del conjunto de entrenamiento X para regresión: {X_reg_train.shape}")
print(f"Tamaño del conjunto de prueba X para regresión: {X_reg_test.shape}")
print(f"Tamaño del conjunto de entrenamiento y para regresión: {y_reg_train.shape}")
print(f"Tamaño del conjunto de prueba y para regresión: {y_reg_test.shape}")

### Paso 7: Construcción y evaluación de un modelo de regresión lineal simple
Utilizaremos la regresión lineal simple, un modelo básico que permite entender la relación lineal entre una única variable independiente y la variable dependiente. Prediremos el  usando solo .

In [ ]:
print("\n--- Modelado con Regresión Lineal Simple ---")

# Seleccionar solo 'annual_inc' como variable independiente para regresión simple
X_reg_simple_train = X_reg_train[['annual_income']]
X_reg_simple_test = X_reg_test[['annual_income']]

# Crear una instancia del modelo de Regresión Lineal
model_reg_simple = LinearRegression()

# Entrenar el modelo con los datos de entrenamiento
model_reg_simple.fit(X_reg_simple_train, y_reg_train)

# Mostrar los coeficientes del modelo
print("Coeficientes del modelo de Regresión Lineal Simple:")
print(f"annual_income: {model_reg_simple.coef_[0]:.4f}")
print(f"Intercepto: {model_reg_simple.intercept_:.4f}")

# Realizar predicciones sobre el conjunto de prueba
y_reg_pred_simple = model_reg_simple.predict(X_reg_simple_test)

# Calcular métricas de evaluación
mse_reg_simple = mean_squared_error(y_reg_test, y_reg_pred_simple)
r2_reg_simple = r2_score(y_reg_test, y_reg_pred_simple)

print(f"\nError Cuadrático Medio (MSE) en el conjunto de prueba (Simple): {mse_reg_simple:.4f}")
print(f"R-cuadrado (R²) en el conjunto de prueba (Simple): {r2_reg_simple:.4f}")

# Visualizar las predicciones vs. los valores reales
plt.figure(figsize=(12, 6))
plt.scatter(y_reg_test, y_reg_pred_simple, alpha=0.7)
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2) # Línea de 45 grados ideal
plt.title('Valores Reales vs. Predicciones del Monto del Préstamo (Regresión Lineal Simple)')
plt.xlabel('Valores Reales del Monto del Préstamo (USD)')
plt.ylabel('Predicciones del Monto del Préstamo (USD)')
plt.grid(True)
plt.show()

**Pregunta de Reflexión 5**:
¿Cómo interpretas el coeficiente de  para la Regresión Lineal Simple en el contexto del monto de un préstamo? ¿Qué te indican el MSE y el R-cuadrado sobre el rendimiento de este modelo con una sola variable? ¿Consideras que este modelo es suficiente para predecir el monto del préstamo en un escenario real de riesgo crediticio?


**Posible Respuesta**:
*   El coeficiente de  (si es positivo) indica que por cada unidad de aumento en el ingreso anual (escalado), el monto del préstamo () tiende a aumentar en la cantidad de ese coeficiente, manteniendo todo lo demás constante.
*   El Error Cuadrático Medio (MSE) mide el promedio de los errores al cuadrado entre las predicciones y los valores reales. Un MSE más bajo indica un mejor ajuste.
*   El R-cuadrado (R²) indica la proporción de la varianza en la variable dependiente () que es predecible a partir de la única variable independiente (). Un valor más cercano a 1 indica un mejor ajuste del modelo a los datos. En una regresión simple, el R² podría ser menor que en modelos multivariables, reflejando que una sola variable no explica toda la variabilidad.
*   En un escenario real de riesgo crediticio, es poco probable que un modelo de regresión lineal simple sea suficiente para predecir el monto del préstamo. El monto del préstamo es influenciado por múltiples factores (historial crediticio, otros créditos, propósito del préstamo, etc.), y una sola variable no capturará adecuadamente esta complejidad. Se necesitan más variables y potencialmente modelos más complejos.




### Paso 8: Construcción y evaluación de un modelo de regresión lineal multivariable  
Ahora utilizaremos la regresión lineal multivariable, que incorpora múltiples variables independientes para predecir la variable dependiente.

In [ ]:
print("\n--- Modelado con Regresión Lineal Multivariable ---")

# Crear una instancia del modelo de Regresión Lineal (usando todas las X_reg_train)
model_reg_multi = LinearRegression()

# Entrenar el modelo con los datos de entrenamiento
model_reg_multi.fit(X_reg_train, y_reg_train)

# Mostrar los coeficientes del modelo (se muestran solo los primeros para concisión)
print("Coeficientes del modelo de Regresión Lineal Multivariable (primeros 5):")
for feature, coef in zip(X_reg_train.columns[:5], model_reg_multi.coef_[:5]):
    print(f"{feature}: {coef:.4f}")
print(f"Intercepto: {model_reg_multi.intercept_:.4f}")

# Realizar predicciones sobre el conjunto de prueba
y_reg_pred_multi = model_reg_multi.predict(X_reg_test)

# Calcular métricas de evaluación
mse_reg_multi = mean_squared_error(y_reg_test, y_reg_pred_multi)
r2_reg_multi = r2_score(y_reg_test, y_reg_pred_multi)

print(f"\nError Cuadrático Medio (MSE) en el conjunto de prueba (Multivariable): {mse_reg_multi:.4f}")
print(f"R-cuadrado (R²) en el conjunto de prueba (Multivariable): {r2_reg_multi:.4f}")

# Visualizar las predicciones vs. los valores reales
plt.figure(figsize=(12, 6))
plt.scatter(y_reg_test, y_reg_pred_multi, alpha=0.7)
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2) # Línea de 45 grados ideal
plt.title('Valores Reales vs. Predicciones del Monto del Préstamo (Regresión Lineal Multivariable)')
plt.xlabel('Valores Reales del Monto del Préstamo (USD)')
plt.ylabel('Predicciones del Monto del Préstamo (USD)')
plt.grid(True)
plt.show()

**Pregunta de Reflexión 6**:
Compara el rendimiento del modelo de Regresión Lineal Multivariable con el de Regresión Lineal Simple (Paso 7). ¿Qué ventajas se observan al usar múltiples variables independientes, y qué implicaciones tiene la interpretabilidad de sus coeficientes en comparación con el modelo simple?


**Posible Respuesta**:
*   La Regresión Lineal Multivariable generalmente tendrá un MSE más bajo y un R-cuadrado más alto que la Regresión Lineal Simple. Esto se debe a que al incorporar más variables independientes, el modelo tiene más información para explicar la variabilidad en la variable objetivo, resultando en predicciones más precisas.
*   Las ventajas de usar múltiples variables independientes incluyen una mayor capacidad explicativa y predictiva. Cada coeficiente indica el cambio esperado en la variable dependiente () por cada unidad de cambio en la variable independiente correspondiente, manteniendo todas las demás variables constantes. Esto permite una comprensión más detallada de los factores que influyen en el monto del préstamo.
*   Sin embargo, la interpretabilidad de cada coeficiente puede volverse más compleja debido a las interacciones y posibles multicolinealidades entre las variables. Mientras que en un modelo simple el coeficiente de  era directo, en un modelo multivariable, su interpretación depende de que "todas las demás variables permanezcan constantes", lo cual es una simplificación en un contexto de negocio.




### Paso 9: Interpretación de Modelos Lineales de Regresión


Para modelos lineales, la interpretación es directa a través de sus coeficientes. Analizaremos los coeficientes del modelo de Regresión Lineal Multivariable para entender la contribución de cada característica a la predicción del funded_amount.

In [ ]:
print("\n--- Interpretación del Modelo de Regresión Lineal Multivariable ---")

# Crear un DataFrame con los coeficientes e identificadores de las características
coef_df_reg = pd.DataFrame({'Feature': X_reg_train.columns, 'Coefficient': model_reg_multi.coef_})
coef_df_reg['Abs_Coefficient'] = np.abs(coef_df_reg['Coefficient'])
coef_df_reg = coef_df_reg.sort_values(by='Abs_Coefficient', ascending=False)

print("Coeficientes del modelo de Regresión Lineal Multivariable (ordenados por magnitud):")
print(coef_df_reg.drop(columns=['Abs_Coefficient']).head(10)) # Mostrar los 10 más influyentes

# Visualización de la importancia de los coeficientes (magnitud)
plt.figure(figsize=(12, 7))
sns.barplot(x='Coefficient', y='Feature', data=coef_df_reg.head(10), palette='coolwarm')
plt.title('Importancia de las Características (Coeficientes) en la Predicción de funded_amount')
plt.xlabel('Valor del Coeficiente')
plt.ylabel('Característica')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

**Pregunta de Reflexión 7**:
Basado en los coeficientes del modelo de Regresión Lineal Multivariable, ¿cuáles son las variables más influyentes en la predicción del  funded_amount y en qué dirección (positiva o negativa) influyen? ¿Cómo puede esta información ser útil para una institución financiera?


**Posible Respuesta**:
*   Las variables con los coeficientes de mayor magnitud (en valor absoluto) son las más influyentes. Un coeficiente positivo indica que un aumento en el valor de esa característica se asocia con un aumento en el  predicho, mientras que un coeficiente negativo sugiere una relación inversa.
*   Por ejemplo, si  tiene un coeficiente positivo y grande, esto significa que a mayor ingreso anual, mayor es el monto del préstamo esperado. Si  (relación deuda-ingresos) tiene un coeficiente negativo, indicaría que a mayor endeudamiento relativo al ingreso, menor sería el monto del préstamo.

Esta información es muy útil para una institución financiera para:
*   **Diseñar políticas de crédito**: Entender qué factores (ej. ingresos, plazo, historial) están más relacionados con el monto de préstamo.
*   **Evaluar solicitudes**: Saber qué características del solicitante impactan más en el monto que se le podría otorgar.
*   **Interpretar decisiones**: Justificar por qué a un cliente se le ofrece un monto particular de préstamo [101].
*   **Identificar áreas de mejora**: Si una variable importante no está siendo bien capturada, se podría mejorar la recolección de datos.



## Parte C: Modelado y Comparación de Clasificación (Nivel de Riesgo Crediticio)  

Ahora, enfocaremos la predicción en una variable categórica: el  del préstamo (éxito/no éxito), que representa el nivel de riesgo crediticio.

### Paso 10: Estrategia de Partición de Datos y Línea Base






Definiremos las variables independientes (X) y la variable dependiente (y) y dividiremos el dataset en conjuntos de entrenamiento y prueba. Para una evaluación más robusta y la optimización de hiperparámetros, introduciremos la Validación Cruzada K-Fold. Como punto de comparación, entrenaremos una Regresión Logística como modelo de referencia (baseline).

In [ ]:
# Definir las variables independientes (X_cls) y la variable dependiente (y_cls)
# Para la clasificación, usaremos todas las características procesadas excepto 'loan_success' (nuestra variable objetivo)
# y 'funded_amount' (la variable objetivo de regresión).
X_cls = df_processed.drop(columns=['loan_success', 'funded_amount']) # Variables de entrada
y_cls = df_processed['loan_success'] # Variable objetivo a clasificar

# Dividir los datos en conjuntos de entrenamiento y prueba para clasificación
# Usaremos el 70% para entrenamiento y el 30% para prueba
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.30, random_state=42)

print(f"\nTamaño del conjunto de entrenamiento X para clasificación: {X_train.shape}")
print(f"Tamaño del conjunto de prueba X para clasificación: {X_test.shape}")
print(f"Tamaño del conjunto de entrenamiento y para clasificación: {y_train.shape}")
print(f"Tamaño del conjunto de prueba y para clasificación: {y_test.shape}")


In [ ]:

# Configuración de Validación Cruzada K-Fold para modelos futuros
cv_strategy = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42) # Usar 5 splits, 3 repeticiones

# --- Regresión Logística como Línea Base (Baseline Model)
print("\n--- Modelado con Regresión Logística (Línea Base) ---")

# Crear una instancia del modelo de Regresión Logística
model_cls_lr = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)

# Entrenar el modelo con los datos de entrenamiento
model_cls_lr.fit(X_train, y_train)

print("Modelo de Regresión Logística entrenado.")

# Realizar predicciones sobre el conjunto de prueba
y_pred_lr = model_cls_lr.predict(X_test)

# Calcular métricas de evaluación para clasificación
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
conf_matrix_lr = confusion_matrix(y_test, y_pred_lr)

print(f"\nAccuracy (Regresión Logística) en el conjunto de prueba: {accuracy_lr:.4f}")
print(f"Precision (Regresión Logística) en el conjunto de prueba: {precision_lr:.4f}")
print(f"Recall (Regresión Logística) en el conjunto de prueba: {recall_lr:.4f}")
print(f"F1-Score (Regresión Logística) en el conjunto de prueba: {f1_lr:.4f}")

print("\nMatriz de Confusión (regresión logística):")
print(conf_matrix_lr)

# Visualizar la matriz de confusión
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicción No Exitoso', 'Predicción Exitoso'],
            yticklabels=['Real No Exitoso', 'Real Exitoso'])
plt.title('Matriz de Confusión (Regresión Logística) - Nivel de Riesgo Crediticio')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

**Pregunta de Reflexión 8**:
¿Por qué es importante establecer un modelo de Regresión Logística como "línea base" en un proyecto de Machine Learning, especialmente al evaluar modelos más complejos? ¿Cómo interpretas las métricas de accuracy, precision,recall  y F1-score para la Regresión Logística en el contexto del riesgo crediticio?


**Posible Respuesta**:
*   Es importante establecer una línea base (baseline), como la Regresión Logística, porque proporciona un punto de referencia simple y, a menudo, interpretable contra el cual comparar el rendimiento de modelos más complejos. Si un modelo avanzado no supera significativamente a la línea base, podría ser preferible el modelo simple por su facilidad de implementación y menor costo computacional.
*   Accuracy: Representa la proporción de predicciones correctas sobre el total de casos. Un valor del 0.8000, por ejemplo, significa que el 80% de las veces, el modelo predice correctamente si un préstamo será exitoso o no.
*   Precision: Proporción de verdaderos positivos sobre el total de predicciones positivas. Indica la confiabilidad cuando el modelo predice la clase positiva (e.g., "Préstamo Exitoso"). Un valor alto significa menos "falsos positivos" (préstamos que se predijeron como exitosos pero no lo fueron).
*   Recall: Proporción de verdaderos positivos sobre el total de casos reales positivos. Indica qué tan bien el modelo identifica todos los casos positivos reales. Un valor alto significa menos "falsos negativos" (préstamos que fueron exitosos pero el modelo predijo que no).
*   F1-Score: Media armónica de Precision y Recall, ofreciendo un equilibrio entre ambas métricas. Es útil cuando hay un desequilibrio entre las clases. En el contexto de riesgo crediticio, un buen F1-Score es deseable, pero la importancia relativa de Precision y Recall puede variar dependiendo de si es más costoso un falso positivo (otorgar un préstamo a alguien que no pagará) o un falso negativo (denegar un préstamo a alguien que sí pagaría).



### Paso 11: Implementación y evaluación del modelo de árbol de decisión


Los Árboles de Decisión son modelos supervisados ampliamente utilizados que pueden resolver problemas de clasificación, siendo la base de algoritmos más complejos.

In [ ]:
print("\n--- Modelado con Árbol de Decisión ---")

# Crear una instancia del modelo de Árbol de Decisión
model_dt = DecisionTreeClassifier(random_state=42, max_depth=5)

# Entrenar el modelo con los datos de entrenamiento
model_dt.fit(X_train, y_train)

print("Modelo de Árbol de Decisión entrenado.")

# Realizar predicciones sobre el conjunto de prueba
y_pred_dt = model_dt.predict(X_test)

# Calcular métricas de evaluación para Árbol de Decisión
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
conf_matrix_dt = confusion_matrix(y_test, y_pred_dt)

print(f"\nAccuracy (Árbol de Decisión) en el conjunto de prueba: {accuracy_dt:.4f}")
print(f"Precision (Árbol de Decisión) en el conjunto de prueba: {precision_dt:.4f}")
print(f"Recall (Árbol de Decisión) en el conjunto de prueba: {recall_dt:.4f}")
print(f"F1-Score (Árbol de Decisión) en el conjunto de prueba: {f1_dt:.4f}")

print("\nMatriz de Confusión (Árbol de Decisión):")
print(conf_matrix_dt)

# Visualizar la matriz de confusión de Árbol de Decisión
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix_dt, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Predicción No Exitoso', 'Predicción Exitoso'],
            yticklabels=['Real No Exitoso', 'Real Exitoso'])
plt.title('Matriz de Confusión (Árbol de Decisión) - Nivel de Riesgo Crediticio')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

Se graficará los árboles como ejemplo de lo que se está evaluando.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20,10))
plot_tree(model_dt,
          feature_names=X_train.columns,
          class_names=['No Exitoso','Exitoso'],
          filled=True,
          rounded=True)
plt.show()

### Paso 12: Optimización y Evaluación de Random Forest
Random Forest es un método de combinación (ensemble method) que utiliza múltiples árboles de decisión para mejorar la capacidad predictiva y de generalización, siendo muy utilizado en la industria financiera. Aplicaremos optimización de hiperparámetros con  y  para encontrar la mejor configuración.

In [ ]:
from tqdm import tqdm
from joblib import parallel_backend
import joblib

# Clase wrapper para mostrar progreso
class TqdmJoblib(object):
    def __init__(self, total=None):
        self._total = total

    def __call__(self, *args, **kwargs):
        return self

    def __enter__(self):
        self._progress_bar = tqdm(total=self._total)
        self._old_batch_callback = joblib.parallel.BatchCompletionCallBack

        class BatchCompletionCallBack(joblib.parallel.BatchCompletionCallBack):
            def __call__(inner_self, *args, **kwargs):
                self._progress_bar.update(n=1)
                return self._old_batch_callback.__call__(inner_self, *args, **kwargs)

        joblib.parallel.BatchCompletionCallBack = BatchCompletionCallBack
        return self._progress_bar

    def __exit__(self, exc_type, exc_value, traceback):
        joblib.parallel.BatchCompletionCallBack = self._old_batch_callback
        self._progress_bar.close()


**Nota**: El siguien bloque de código tarda varios minutos (3 a 5 minutos) ya que cada combinación implica entrenar varios árboles con validación cruzada y el espacio de búsqueda puede ser grande si no se define correctamente y tardar horas. Por el momento son 5 × 2 = 10 entrenamientos completos de Random Forest.

In [ ]:

print("\n--- Modelado y Optimización con Random Forest ---")


# Configurar RandomizedSearchCV normalmente+
# Crear una instancia de RandomizedSearchCV
# n_iter = 5: número de combinaciones de parámetros a probar
# cv = cv_strategy: se puede usar la estrategia de validación cruzada definida anteriormente pero aumenta las posibilidades y aumenta el tiempo
# scoring = 'f1': métrica para evaluar el rendimiento (f1 es bueno para clases desbalanceadas)
rf = RandomForestClassifier(random_state=42)
# Definir el espacio de parámetros a buscar para Random Forest
# Se define un rango de valores para cada hiperparámetro relevante
param_dist = {
    "n_estimators": [50, 100, 200],# Número de árboles en el bosque
    #'max_features': ['sqrt', 'log2', None], # Número de características a considerar para la mejor división
    "max_depth": [None, 5, 10],# Profundidad máxima del árbol
    "min_samples_split": [2, 5], # Número mínimo de muestras requeridas para dividir un nodo interno
    "min_samples_leaf": [1, 2]# Número mínimo de muestras requeridas en cada nodo hoja
    #'bootstrap': [True, False] # Si se utilizan muestras de arranque al construir árboles
}



randomized_search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=5,
    cv=2,
    n_jobs=-1,
    random_state=42
)

# Calcular número de fits totales
total_fits = randomized_search_rf.n_iter * randomized_search_rf.cv

with TqdmJoblib(total=total_fits) as progress_bar:
    # Ejecutar la búsqueda aleatoria sobre los datos de entrenamiento
    randomized_search_rf.fit(X_train, y_train)

In [ ]:
# Obtener el mejor modelo encontrado
best_model_rf = randomized_search_rf.best_estimator_
print(f"\nMejores hiperparámetros para Random Forest: {randomized_search_rf.best_params_}")
print(f"Mejor score de validación (F1-Score) para Random Forest: {randomized_search_rf.best_score_:.4f}")

# Evaluar el mejor modelo en el conjunto de prueba
y_pred_rf = best_model_rf.predict(X_test)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)

print(f"\nAccuracy (Random Forest Optimizado) en el conjunto de prueba: {accuracy_rf:.4f}")
print(f"Precision (Random Forest Optimizado) en el conjunto de prueba: {precision_rf:.4f}")
print(f"Recall (Random Forest Optimizado) en el conjunto de prueba: {recall_rf:.4f}")
print(f"F1-Score (Random Forest Optimizado) en el conjunto de prueba: {f1_rf:.4f}")

print("\nMatriz de Confusión (Random Forest Optimizado):")
print(conf_matrix_rf)

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix_rf, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Predicción No Exitoso', 'Predicción Exitoso'],
            yticklabels=['Real No Exitoso', 'Real Exitoso'])
plt.title('Matriz de Confusión (Random Forest Optimizado) - Nivel de Riesgo Crediticio')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

### Paso 13: Optimización y Evaluación de Red Neuronal (Perceptrón Multicapa - MLPClassifier)
Las Redes Neuronales (MLP) son modelos no lineales avanzados que pueden capturar relaciones complejas en los datos. También aplicaremos la optimización de hiperparámetros para este modelo.

In [ ]:
print("\n--- Modelado y Optimización con Red Neuronal (MLPClassifier) ---")

# Definir el espacio de parámetros a buscar para MLPClassifier
grid_params_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)], # Número de neuronas en las capas ocultas
    'activation': ['relu', 'tanh'], # Función de activación para las capas ocultas
    'solver': ['adam', 'sgd'], # Algoritmo para optimización de pesos [Previous Response]
    'alpha': [0.0001, 0.001, 0.01], # Término de regularización L2 (evita sobreajuste)
    'learning_rate_init': [0.001, 0.01], # Tasa de aprendizaje inicial
    'max_iter': [114] # Número máximo de iteraciones [Previous Response]
}

# Crear una instancia de RandomizedSearchCV para MLPClassifier
# Nota: La búsqueda de hiperparámetros para Redes Neuronales puede ser muy lenta.
# Se reduce n_iter para propósitos prácticos del proyecto.
randomized_search_mlp = RandomizedSearchCV(
    estimator=MLPClassifier(random_state=42),
    param_distributions=grid_params_mlp,
    n_iter=5, # Reducido significativamente para una ejecución más rápida
    cv=2,
    scoring='f1', # O 'accuracy'
    verbose=1,
    random_state=42,
    n_jobs=-1 # Usar todos los cores disponibles
)


Nota: El siguien bloque de código tarda varios minutos (33 a 70 minutos aprox.) ya que cada combinación implica entrenar varios árboles con validación cruzada y el espacio de búsqueda puede ser grande si no se define correctamente y tardar horas. Por el momento son 5 × 2 = 10 entrenamientos completos de Random Forest.

In [ ]:
# Calcular número total de fits
total_fits = randomized_search_mlp.n_iter * randomized_search_mlp.cv

# Ejecutar con barra de progreso
with TqdmJoblib(total=total_fits) as progress_bar:
    # Ejecutar la búsqueda aleatoria sobre los datos de entrenamiento
    randomized_search_mlp.fit(X_train, y_train)

# Guardar el mejor modelo
best_model_mlp = randomized_search_mlp.best_estimator_
print("\nMejor modelo encontrado:", best_model_mlp)

In [ ]:

print(f"\nMejores hiperparámetros para Red Neuronal (MLP): {randomized_search_mlp.best_params_}")
print(f"Mejor score de validación (F1-Score) para Red Neuronal (MLP): {randomized_search_mlp.best_score_:.4f}")

# Evaluar el mejor modelo en el conjunto de prueba
y_pred_mlp = best_model_mlp.predict(X_test)

accuracy_mlp = accuracy_score(y_test, y_pred_mlp)
precision_mlp = precision_score(y_test, y_pred_mlp)
recall_mlp = recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)
conf_matrix_mlp = confusion_matrix(y_test, y_pred_mlp)

print(f"\nAccuracy (MLPClassifier Optimizado) en el conjunto de prueba: {accuracy_mlp:.4f}")
print(f"Precision (MLPClassifier Optimizado) en el conjunto de prueba: {precision_mlp:.4f}")
print(f"Recall (MLPClassifier Optimizado) en el conjunto de prueba: {recall_mlp:.4f}")
print(f"F1-Score (MLPClassifier Optimizado) en el conjunto de prueba: {f1_mlp:.4f}")

print("\nMatriz de Confusión (MLPClassifier Optimizado):")
print(conf_matrix_mlp)

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix_mlp, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicción No Exitoso', 'Predicción Exitoso'],
            yticklabels=['Real No Exitoso', 'Real Exitoso'])
plt.title('Matriz de Confusión (MLPClassifier Optimizado) - Nivel de Riesgo Crediticio')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()


### Paso 14: Interpretación de Modelos No Lineales (SHAP)


Los modelos no lineales, como Random Forest o Redes Neuronales, ofrecen un alto rendimiento pero su interpretabilidad es un desafío. Utilizaremos SHAP (Shapley Additive Explanations) para entender la contribución de las variables en las predicciones individuales y globales. Escogeremos el Random Forest Optimizado como ejemplo para esta interpretación, ya que XGBoost no se ha implementado en esta versión.

In [ ]:
# Asegurar datos numéricos
X_train_numeric = X_train.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
X_test_numeric = X_test.apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

# Convertir booleanos a enteros
X_train_numeric = X_train_numeric.astype({col: 'int' for col in X_train_numeric.select_dtypes('bool').columns})
X_test_numeric = X_test_numeric.astype({col: 'int' for col in X_test_numeric.select_dtypes('bool').columns})

# Tomar una instancia
target_datapoint_index_shap = 0
X_test_instance = X_test_numeric.iloc[[target_datapoint_index_shap]]

# Background dataset
background_distribution_shap = shap.sample(X_train_numeric, 50)

# Explainer
#explainer_rf = shap.TreeExplainer(best_model_rf, data=background_distribution, model_output="probability")
explainer_rf = shap.TreeExplainer(best_model_rf, data=background_distribution_shap)
shap_values_rf = explainer_rf.shap_values(X_test_instance)

# Predicción y expected_value
pred_class = best_model_rf.predict(X_test_instance)[0]
base_value = explainer_rf.expected_value[pred_class] if isinstance(explainer_rf.expected_value, np.ndarray) else explainer_rf.expected_value

# checamos qué tipo de salida es shap_values_rf
print(type(shap_values_rf))

# shap_values_rf ya es ndarray con shape (1, n_features)
#values = shap_values_rf[0]

# Extraer los shap_values para la instancia seleccionada
raw_values = shap_values_rf[0]

# Si es 2D (multi-output), elegir la primera salida
if raw_values.ndim > 1:
    values = raw_values[:, 0]
else:
    values = raw_values


shap.plots.waterfall(
    shap.Explanation(
        values=values,
        base_values=base_value,
        data=X_test_instance.iloc[0],
        feature_names=X_test_numeric.columns.tolist()
    )
)

Nota: El siguien bloque de código tarda varios minutos (5 minutos aprox.) calcular SHAP para todo el conjunto de prueba con Random Forest puede ser muy lento porque SHAP está evaluando cada árbol y cada instancia. Para un dataset grande como este tarda demasiado y casi siempre no se necesita los SHAP de todo el X_test para un waterfall de una sola instancia

In [ ]:
# Solo la instancia 86
X_instance = X_test.iloc[[target_datapoint_index_shap]]
shap_values_rf = explainer_rf(X_instance)

# Seleccionar la clase positiva (columna 1)
values = shap_values_rf.values[0][:, 1]   # (n_features,)
base_value = shap_values_rf.base_values[0][1]
data_instance = X_instance.iloc[0]

# Explicación
exp = shap.Explanation(
    values=values,
    base_values=base_value,
    data=data_instance,
    feature_names=X_test.columns.tolist()
)

shap.plots.waterfall(exp)


**Pregunta de Reflexión 9**:
Explica la interpretación de los gráficos SHAP para el modelo Random Forest. ¿Cómo SHAP te ayuda a comprender las decisiones de un modelo de "caja negra" en el contexto del riesgo crediticio, y por qué esta interpretabilidad es crucial para una institución financiera?

**Posible Respuesta**:
*   El gráfico de cascada (waterfall plot) para una instancia individual muestra cómo las diferentes características () contribuyen a llevar la predicción del modelo desde el valor base () hasta la predicción final para esa instancia específica. Las barras que van a la derecha aumentan la probabilidad de la clase positiva (ej. "Préstamo Exitoso"), y las que van a la izquierda la disminuyen. Esto nos permite ver, por ejemplo, que un  alto contribuyó positivamente a que la predicción fuera "Exitoso", mientras que un  alto contribuyó negativamente.
*   El gráfico de enjambre de abejas (beeswarm plot) proporciona una visión global de la importancia de cada característica y su efecto en el conjunto de datos. Cada punto representa una instancia del dataset, y su posición horizontal indica la magnitud del impacto SHAP, mientras que el color suele indicar el valor de la característica para esa instancia (ej. rojo = valor alto, azul = valor bajo). Esto permite identificar las características más importantes en general (las que tienen mayor dispersión vertical) y cómo sus valores altos o bajos afectan la predicción (ej.  alto tiende a aumentar la probabilidad de éxito,  alto tiende a disminuirla).
*   SHAP es crucial para una institución financiera porque permite interpretar modelos de "caja negra" (como Random Forest o Redes Neuronales) que, a pesar de su alto rendimiento, son difíciles de entender directamente. Esta interpretabilidad es esencial para:
*   **Justificación de decisiones**: Poder explicar a un solicitante por qué su préstamo fue aprobado o denegado [126].
*   **Cumplimiento regulatorio**: Muchas regulaciones financieras exigen transparencia en los modelos utilizados.
*   **Confianza y adopción**: Generar confianza entre los usuarios y los equipos de negocio en las decisiones del modelo [101, 126].
*   **Identificación de sesgos**: Detectar si el modelo está tomando decisiones basadas en características injustas o sesgadas.

### Paso 15: Comparación Integral de Modelos de Clasificación


Finalmente, resumiremos y compararemos el rendimiento de todos los modelos de clasificación implementados para el riesgo crediticio.

In [ ]:
# --- Comparación Resumida de Todos los Modelos de Clasificación ---
print("\n--- Comparación Integral de Modelos de Clasificación (Nivel de Riesgo Crediticio) ---")

comparison_data_cls = {
    'Modelo': ['Regresión Logística', 'Árbol de Decisión', 'Random Forest Optimizado', 'Red Neuronal (MLP) Optimizado'],
    'Accuracy': [accuracy_lr, accuracy_dt, accuracy_rf, accuracy_mlp],
    'Precision': [precision_lr, precision_dt, precision_rf, precision_mlp],
    'Recall': [recall_lr, recall_dt, recall_rf, recall_mlp],
    'F1-Score': [f1_lr, f1_dt, f1_rf, f1_mlp]
}

comparison_df_cls = pd.DataFrame(comparison_data_cls)
print(comparison_df_cls.set_index('Modelo'))

# Visualización de la comparación de métricas
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
axes_flat = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sns.barplot(x='Modelo', y=metric, data=comparison_df_cls, ax=axes_flat[i], palette='viridis')
    axes_flat[i].set_title(f'Comparación de {metric}')
    axes_flat[i].set_ylim(0.5, 1.0) # Ajustar límites para mejor visualización
    axes_flat[i].tick_params(axis='x', rotation=45)
    axes_flat[i].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


**Pregunta de Reflexión 10**:

Considerando todos los modelos de clasificación evaluados (Regresión Logística, Árbol de Decisión, Random Forest, Red Neuronal/MLP) y sus respectivas métricas, ¿cuál modelo recomendarías para predecir el nivel de riesgo crediticio en un entorno financiero real y por qué? Justifica tu respuesta considerando el balance entre rendimiento predictivo, interpretabilidad, costos computacionales y las implicaciones de los diferentes tipos de errores (falsos positivos, falsos negativos) en este contexto.


Posible Respuesta:
*   Al comparar las métricas, es probable que Random Forest o la Red Neuronal (MLP) muestren un mejor rendimiento predictivo (mayor Accuracy, Precision, Recall, F1-Score) que la Regresión Logística y el Árbol de Decisión individual. Esto se debe a su capacidad para capturar relaciones no lineales y más complejas en los datos.
*   La Regresión Logística y el Árbol de Decisión individual son más simples y, por ende, más interpretables. Sus reglas o coeficientes son más transparentes, lo cual es ventajoso para justificar decisiones ante clientes o reguladores.
*   Costos computacionales: La Regresión Logística y el Árbol de Decisión son generalmente menos costosos de entrenar que Random Forest y MLP, especialmente cuando se realiza optimización de hiperparámetros. Las Redes Neuronales pueden ser particularmente costosas.

Implicaciones de errores:
*   Un **Falso Positivo (FP)** en riesgo crediticio significa clasificar a un cliente como "Exitoso" cuando en realidad no lo será (se le otorga un préstamo que incumplirá). Esto genera una pérdida financiera para el banco. Priorizar la **Precision** ayuda a reducir los FPs.
*   Un **Falso Negativo (FN)** significa clasificar a un cliente como "No Exitoso" cuando en realidad sí pagaría (se le deniega un préstamo a un buen cliente). Esto representa una oportunidad de negocio perdida. Priorizar el **Recall** ayuda a reducir los FNs.
*   Recomendación:
*   Si la **prioridad es maximizar el rendimiento predictivo** y la capacidad de capturar relaciones complejas, y se pueden invertir recursos en interpretar el modelo (usando SHAP, como se demostró), entonces un **Random Forest Optimizado** o una **Red Neuronal (MLP) Optimizado** serían la mejor elección. Random Forest a menudo ofrece un buen balance entre rendimiento y relativa interpretabilidad (comparado con MLP).
*   Si la **prioridad es la interpretabilidad y la simplicidad** a toda costa, incluso con un rendimiento ligeramente inferior, la **Regresión Logística** podría ser preferible como modelo de línea base o para complementar un modelo más complejo. Para el riesgo crediticio, donde la justificación es clave, una solución híbrida (un modelo complejo para la predicción y un modelo simple o SHAP para la explicación) es a menudo lo ideal.
*   Para una institución financiera, un modelo con un alto **F1-Score** y un buen balance entre **Precision** y **Recall** (quizás priorizando ligeramente Precision para evitar pérdidas por incumplimiento) sería crucial. Un **Random Forest** a menudo es una excelente opción intermedia, ofreciendo alto rendimiento y la capacidad de explicar sus decisiones con SHAP.


<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>